# AIC2026 — Kaggle Notebook: Generate All Vector Embeddings (GPU)

**Mục đích:** Sinh đồng thời image/ocr/asr embeddings trên GPU Kaggle.

**Đầu ra:** File `unified_dataset.zip` tải về máy local nạp vào Qdrant DB.

In [ ]:
# 1. Install dependencies (Khóa numpy<2.0 để tránh lỗi numpy 2.x incompatible với scipy/transformers)
!pip install -q "transformers>=4.40.0" "numpy<2.0" accelerate sentencepiece einops timm

import os, sys
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone Private Repo & Dynamic Module Reload (Xóa cache Python kernel an toàn)
import os, sys, shutil, importlib
from pathlib import Path

# Luôn chuyển về thư mục gốc /kaggle/working trước để tránh lỗi rmtree xóa thư mục đang đứng
os.chdir("/kaggle/working")

GITHUB_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
    print("🔑 Lấy thành công GITHUB_TOKEN từ Kaggle Secrets!")
except Exception:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

repo_dir = Path("/kaggle/working/AIC2026").absolute()

# Xóa folder cũ để luôn pull code mới nhất từ GitHub
if repo_dir.exists():
    shutil.rmtree(repo_dir, ignore_errors=True)

if GITHUB_TOKEN:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/ThanhDanh1510/Super-Image-Retrieval-System-for-AIC2026.git"
else:
    repo_url = "https://github.com/ThanhDanh1510/Super-Image-Retrieval-System-for-AIC2026.git"
print("Executing fresh git clone...")
!git clone {repo_url} {repo_dir}

os.chdir(str(repo_dir))
sys.path.extend([str(repo_dir), "/kaggle/working", "."])

# Ép Python xóa cache module cũ trong memory
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("shared") or mod_name.startswith("ingestion"):
        del sys.modules[mod_name]

from shared.models.image_embedding_wrapper import ImageEmbeddingWrapper
from shared.models.text_embedding_wrapper import TextEmbeddingWrapper

embed_mod = importlib.import_module("ingestion.skeletons.01c_embedding_generation")
run_embedding_generation = embed_mod.run_embedding_generation
print("✅ UNLOADED CACHE & IMPORT SUCCESSFUL!")

In [ ]:
# 3. Thiết lập đường dẫn Input & Output (Tự động tìm đệ quy thư mục chứa scene_metadata.parquet)
import sys, os, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path("/kaggle/working/unified_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_dir = None
# Đệ quy tìm thư mục chứa file scene_metadata.parquet trong /kaggle/input/
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").rglob("scene_metadata.parquet"):
        input_dir = p.parent
        break

if not input_dir:
    for p in [Path("/kaggle/input/datasets/danhhcmus/supa-data-imagelol/unified_raw"), Path("/kaggle/input/aic2026-unified-raw"), Path("/kaggle/working/unified_raw")]:
        if p.exists():
            input_dir = p
            break

print(f"📌 INPUT DIR FOUND: {input_dir}")
assert input_dir and input_dir.exists(), "Không tìm thấy thư mục input chứa scene_metadata.parquet!"

In [ ]:
# 4. Chạy Pipeline sinh Embeddings bằng GPU
import sys, os, importlib
from pathlib import Path
repo_dir = Path("/kaggle/working/AIC2026").absolute()
for p in [str(repo_dir), "/kaggle/working", "."]:
    if p not in sys.path:
        sys.path.insert(0, p)

metadata_path = str(input_dir / "scene_metadata.parquet")
ocr_path = str(input_dir / "ocr_text.parquet")
asr_path = str(input_dir / "asr_text.parquet")
keyframe_dir = str(input_dir / "keyframes")

run_embedding_generation(
    metadata_path=metadata_path,
    ocr_path=ocr_path,
    asr_path=asr_path,
    keyframe_dir=keyframe_dir,
    output_dir=str(OUTPUT_DIR),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [ ]:
# 5. Copy parquets và đóng gói thành unified_dataset.zip
import shutil, os, zipfile
from pathlib import Path

shutil.copy2(metadata_path, OUTPUT_DIR / "scene_metadata.parquet")
shutil.copy2(ocr_path, OUTPUT_DIR / "ocr_text.parquet")
shutil.copy2(asr_path, OUTPUT_DIR / "asr_text.parquet")

zip_filename = "/kaggle/working/unified_dataset.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            zipf.write(file, arcname=file.relative_to(OUTPUT_DIR))

print(f"✅ HOÀN THÀNH VÀ ĐÓNG GÓI: {zip_filename}")
print(f"Size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")